# 06 · Schedulers

> **Source notes:** `Schedulers.md`

Swap the sampler → cut generation time by 50× **without retraining**.

This notebook demonstrates:
- Visualise the DDPM noise schedule (ᾱₜ curve)
- Implement DDIM sampling on our Ch.4/Ch.5 `CondUNet`
- Compare image quality vs. step count for DDPM and DDIM
- Profile wall-clock time per step count
- Visualise how timestep sub-sequences differ (uniform vs. SNR-optimal)

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install",
                "torch", "torchvision", "matplotlib", "numpy", "-q"], check=True)

import torch, torch.nn as nn, torch.nn.functional as F, torchvision
import torchvision.transforms as T
import numpy as np, matplotlib.pyplot as plt, time
from torch.utils.data import DataLoader
print("Ready.")

## 1 · Noise Schedule Visualisation

In [ ]:
T_STEPS = 1000

# Linear schedule (default DDPM)
betas_linear    = torch.linspace(1e-4, 0.02, T_STEPS)
alphas_linear   = 1 - betas_linear
alpha_bar_linear = torch.cumprod(alphas_linear, 0)

# Cosine schedule (improved DDPM, Nichol & Dhariwal 2021)
steps = torch.arange(T_STEPS + 1)
f     = torch.cos((steps / T_STEPS + 0.008) / 1.008 * (torch.pi / 2)) ** 2
alpha_bar_cosine = f / f[0]
alpha_bar_cosine = alpha_bar_cosine[1:]   # same length as linear

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

t_range = torch.arange(T_STEPS)
ax1.plot(t_range, alpha_bar_linear, label="Linear β schedule", color="steelblue")
ax1.plot(t_range, alpha_bar_cosine, label="Cosine schedule", color="tomato", linestyle="--")
ax1.set_xlabel("Timestep t"); ax1.set_ylabel("ᾱₜ (signal fraction²)")
ax1.set_title("Noise Schedule: Linear vs. Cosine")
ax1.legend(); ax1.grid(alpha=0.3)

# SNR = ᾱ / (1-ᾱ) — log scale shows where gradient signal lives
snr_linear = alpha_bar_linear / (1 - alpha_bar_linear + 1e-8)
snr_cosine = alpha_bar_cosine / (1 - alpha_bar_cosine + 1e-8)
ax2.semilogy(t_range, snr_linear, label="Linear", color="steelblue")
ax2.semilogy(t_range, snr_cosine, label="Cosine", color="tomato", linestyle="--")
ax2.set_xlabel("Timestep t"); ax2.set_ylabel("SNR (log scale)")
ax2.set_title("Signal-to-Noise Ratio")
ax2.legend(); ax2.grid(alpha=0.3)

plt.suptitle("Cosine schedule gives more uniform SNR across timesteps")
plt.tight_layout(); plt.show()

print("Linear: high-SNR region (t<200) spans only 20% of timesteps")
print("Cosine: SNR transitions more gradually → better use of training budget")

## 2 · Rebuild the Conditional U-Net (from Ch.5)

We copy the model architecture and train quickly (15 epochs) for comparison demos.

In [ ]:
betas     = betas_linear
alphas    = alphas_linear
alpha_bar = alpha_bar_linear
sqrt_ab   = alpha_bar.sqrt()
sqrt_1mab = (1 - alpha_bar).sqrt()

NUM_CLASSES, NULL_CLASS = 10, 10

class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    def forward(self, t):
        half  = self.dim // 2
        freqs = torch.exp(-torch.arange(half, device=t.device) * (np.log(10000) / (half-1)))
        args  = t[:,None].float() * freqs[None]
        return torch.cat([args.sin(), args.cos()], dim=-1)

class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_dim):
        super().__init__()
        self.conv1  = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.conv2  = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.t_proj = nn.Linear(time_dim, out_ch)
        self.skip   = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
    def forward(self, x, t_emb):
        h = F.silu(self.conv1(x))
        h = h + self.t_proj(t_emb)[:, :, None, None]
        return F.silu(self.conv2(h)) + self.skip(x)

class CondUNet(nn.Module):
    def __init__(self, base_ch=32, time_dim=64):
        super().__init__()
        self.class_embed = nn.Embedding(NUM_CLASSES + 1, time_dim)
        self.time_embed  = nn.Sequential(
            SinusoidalTimeEmbedding(time_dim),
            nn.Linear(time_dim, time_dim * 4), nn.SiLU(),
            nn.Linear(time_dim * 4, time_dim))
        C = base_ch
        self.enc1 = ResBlock(1,   C,   time_dim)
        self.enc2 = ResBlock(C,   C*2, time_dim)
        self.enc3 = ResBlock(C*2, C*4, time_dim)
        self.bot  = ResBlock(C*4, C*4, time_dim)
        self.dec3 = ResBlock(C*8, C*2, time_dim)
        self.dec2 = ResBlock(C*4, C,   time_dim)
        self.dec1 = ResBlock(C*2, C,   time_dim)
        self.out  = nn.Conv2d(C, 1, 1)
        self.down = nn.MaxPool2d(2)
        self.up   = nn.Upsample(scale_factor=2, mode="nearest")
    def forward(self, x, t, c):
        t_emb = self.time_embed(t) + self.class_embed(c)
        s1 = self.enc1(x, t_emb)
        s2 = self.enc2(self.down(s1), t_emb)
        s3 = self.enc3(self.down(s2), t_emb)
        b  = self.bot(s3, t_emb)
        d3 = self.dec3(torch.cat([self.up(b), s3], 1), t_emb)
        d2 = self.dec2(torch.cat([self.up(d3), s2], 1), t_emb)
        d1 = self.dec1(torch.cat([d2, s1], 1), t_emb)
        return self.out(d1)

model  = CondUNet()
optim  = torch.optim.Adam(model.parameters(), lr=2e-4)
dataset = torchvision.datasets.MNIST(
    "/tmp/mnist", train=True, download=True,
    transform=T.Compose([T.ToTensor(), T.Lambda(lambda x: x*2-1)]))
loader = DataLoader(dataset, batch_size=256, shuffle=True, num_workers=0)

def q_sample(x0, t):
    eps = torch.randn_like(x0)
    return sqrt_ab[t].view(-1,1,1,1)*x0 + sqrt_1mab[t].view(-1,1,1,1)*eps, eps

model.train()
for epoch in range(20):
    for x0, c in loader:
        t = torch.randint(0, T_STEPS, (x0.shape[0],))
        xt, eps = q_sample(x0, t)
        mask = torch.rand(x0.shape[0]) < 0.10
        c_in = c.clone(); c_in[mask] = NULL_CLASS
        loss = F.mse_loss(model(xt, t, c_in), eps)
        optim.zero_grad(); loss.backward(); optim.step()
    if (epoch+1) % 5 == 0:
        print(f"Epoch {epoch+1}/20  loss={loss.item():.4f}")
print("Training done.")

## 3 · DDIM Sampler Implementation

In [ ]:
@torch.no_grad()
def ddim_sample(model, classes, n_steps=50, guidance_scale=3.0, eta=0.0):
    """
    DDIM sampler.
    eta=0 → fully deterministic ODE
    eta=1 → equivalent to DDPM stochastic
    """
    model.eval()
    N    = len(classes)
    null = torch.full((N,), NULL_CLASS, dtype=torch.long)
    x    = torch.randn(N, 1, 28, 28)
    
    # Build sub-sequence: n_steps uniformly spaced timesteps
    ts = torch.linspace(T_STEPS - 1, 0, n_steps + 1).long()
    
    for i in range(n_steps):
        t_cur  = ts[i]
        t_next = ts[i + 1]
        tb     = torch.full((N,), t_cur, dtype=torch.long)
        
        # CFG
        eps_c  = model(x, tb, classes)
        eps_u  = model(x, tb, null)
        eps    = eps_u + guidance_scale * (eps_c - eps_u)
        
        ab_cur  = alpha_bar[t_cur]
        ab_next = alpha_bar[t_next] if t_next >= 0 else torch.tensor(1.0)
        
        # Predict x0
        x0_pred = ((x - sqrt_1mab[t_cur] * eps) / sqrt_ab[t_cur]).clamp(-1, 1)
        
        # DDIM step
        sigma   = eta * ((1 - ab_next) / (1 - ab_cur) * (1 - ab_cur / ab_next)).sqrt()
        x       = (ab_next.sqrt() * x0_pred +
                   (1 - ab_next - sigma**2).clamp(min=0).sqrt() * eps +
                   sigma * torch.randn_like(x))
    return x

@torch.no_grad()
def ddpm_sample(model, classes, guidance_scale=3.0):
    """Original DDPM sampler (1000 steps)."""
    model.eval()
    N    = len(classes)
    null = torch.full((N,), NULL_CLASS, dtype=torch.long)
    x    = torch.randn(N, 1, 28, 28)
    for t_idx in reversed(range(T_STEPS)):
        tb     = torch.full((N,), t_idx, dtype=torch.long)
        eps_c  = model(x, tb, classes)
        eps_u  = model(x, tb, null)
        eps    = eps_u + guidance_scale * (eps_c - eps_u)
        x0_p   = ((x - sqrt_1mab[t_idx] * eps) / sqrt_ab[t_idx]).clamp(-1, 1)
        if t_idx > 0:
            at  = alphas[t_idx]; bt = betas[t_idx]; ab_p = alpha_bar[t_idx-1]
            mu  = (ab_p.sqrt()*bt/(1-alpha_bar[t_idx]))*x0_p + \
                  (at.sqrt()*(1-ab_p)/(1-alpha_bar[t_idx]))*x
            x   = mu + (bt*(1-ab_p)/(1-alpha_bar[t_idx])).sqrt()*torch.randn_like(x)
        else:
            x = x0_p
    return x

print("Samplers defined.")

## 4 · Speed vs. Quality: Step Count Comparison

In [ ]:
classes = torch.tensor([3, 3, 3, 3])  # generate '3' four times
step_counts = [5, 10, 20, 50, 100]

results = {}
for n in step_counts:
    t0 = time.time()
    imgs = ddim_sample(model, classes, n_steps=n, guidance_scale=3.0)
    elapsed = time.time() - t0
    results[n] = (imgs, elapsed)
    print(f"DDIM {n:4d} steps: {elapsed:.2f}s")

# Also run full DDPM for reference
t0 = time.time()
ddpm_imgs = ddpm_sample(model, classes, guidance_scale=3.0)
ddpm_time = time.time() - t0
print(f"DDPM 1000 steps: {ddpm_time:.2f}s")

# Display
n_cols = len(classes)
all_configs = step_counts + ["DDPM-1000"]
fig, axes = plt.subplots(len(all_configs), n_cols, figsize=(n_cols*2, len(all_configs)*2.2))

for row, config in enumerate(all_configs):
    if config == "DDPM-1000":
        imgs, elapsed = ddpm_imgs, ddpm_time
        label = f"DDPM-1000\n{elapsed:.1f}s"
    else:
        imgs, elapsed = results[config]
        label = f"DDIM-{config}\n{elapsed:.1f}s"
    for col in range(n_cols):
        axes[row, col].imshow(imgs[col, 0].numpy(), cmap="gray", vmin=-1, vmax=1)
        axes[row, col].axis("off")
    axes[row, 0].set_ylabel(label, rotation=0, labelpad=55, va="center", fontsize=8)

plt.suptitle("DDIM step count vs. DDPM 1000-step baseline (digit '3')", y=1.01)
plt.tight_layout(); plt.show()

## 5 · Determinism: Same Seed → Same Image

In [ ]:
# DDIM is deterministic: same seed reproduces exactly
classes_single = torch.tensor([7])

results_det = []
for run in range(4):
    torch.manual_seed(42)  # same seed each time
    img = ddim_sample(model, classes_single, n_steps=20, guidance_scale=3.0, eta=0.0)
    results_det.append(img)

# Check they are pixel-identical
max_diff = max(abs(results_det[i] - results_det[0]).max().item() for i in range(1, 4))
print(f"Max pixel difference across 4 runs with same seed: {max_diff:.6f}")

# Now stochastic (eta=1)
results_stoch = []
for run in range(4):
    torch.manual_seed(42)
    img = ddim_sample(model, classes_single, n_steps=20, guidance_scale=3.0, eta=1.0)
    results_stoch.append(img)

# These should differ because of internal randn calls
max_diff_stoch = max(abs(results_stoch[i] - results_stoch[0]).max().item() for i in range(1, 4))

fig, axes = plt.subplots(2, 4, figsize=(8, 4))
for col, img in enumerate(results_det):
    axes[0, col].imshow(img[0,0].numpy(), cmap="gray", vmin=-1, vmax=1)
    axes[0, col].set_title(f"Run {col+1}"); axes[0, col].axis("off")
axes[0, 0].set_ylabel("η=0 (det.)", rotation=0, labelpad=50)
for col, img in enumerate(results_stoch):
    axes[1, col].imshow(img[0,0].numpy(), cmap="gray", vmin=-1, vmax=1)
    axes[1, col].set_title(f"Run {col+1}"); axes[1, col].axis("off")
axes[1, 0].set_ylabel("η=1 (stoch.)", rotation=0, labelpad=50)

plt.suptitle(f"Deterministic (η=0): max diff = {max_diff:.2e}  |  Stochastic (η=1): max diff = {max_diff_stoch:.2e}")
plt.tight_layout(); plt.show()

## 6 · Summary

```
Scheduler decision tree:

  Need real-time (<1s)?  ──Yes──▶ LCM / SDXL-Turbo (needs distilled model)
         │No
         ▼
  Need highest quality?  ──Yes──▶ DPM-Solver++ (15-20 steps)
         │No
         ▼
  Need reproducibility?  ──Yes──▶ DDIM (η=0, 20-50 steps)
         │No
         ▼
  Need maximum diversity ──Yes──▶ DDPM or DDIM (η=1, 50+ steps)
```

**Next:** [LatentDiffusion.md](../LatentDiffusion/LatentDiffusion.md) — bring diffusion to real images by first compressing to latent space.